# 🔍 RAG with LangChain — Hands-On Examples

> **Module 03 | Submodule 07 | Practical Notebook**
>
> Sections:
> 1. Simple RAG pipeline from scratch (manual LCEL)
> 2. create_retrieval_chain API
> 3. Source attribution & streaming
> 4. Conversational RAG with history
> 5. RunnableWithMessageHistory (session management)
> 6. Complete PDF Chatbot

In [ ]:
!pip install langchain langchain-openai langchain-community langchain-chroma \
             langchain-text-splitters faiss-cpu pypdf python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "module-03-rag"

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.documents import Document

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm        = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("Ready!")

In [ ]:
# Sample knowledge base — replace with real PDF/web content
knowledge_base = [
    Document(page_content="LangChain is a Python framework for building LLM-powered applications. It provides abstractions for models, prompts, chains, and agents.", metadata={"source": "langchain_guide.pdf", "page": 1}),
    Document(page_content="LangGraph extends LangChain with stateful, graph-based agent orchestration. Nodes are Python functions and edges define control flow.", metadata={"source": "langchain_guide.pdf", "page": 2}),
    Document(page_content="LangSmith is the observability platform for LangChain. It provides traces, evaluations, and monitoring dashboards for production apps.", metadata={"source": "langchain_guide.pdf", "page": 3}),
    Document(page_content="LCEL (LangChain Expression Language) uses the pipe | operator to compose Runnables. Every component is a Runnable with invoke, stream, and batch methods.", metadata={"source": "lcel_guide.pdf", "page": 1}),
    Document(page_content="RAG (Retrieval-Augmented Generation) grounds LLM answers in real documents. The pipeline: Load → Split → Embed → Store → Retrieve → Generate.", metadata={"source": "rag_guide.pdf", "page": 1}),
    Document(page_content="ChromaDB is an open-source vector database offering persistent storage and metadata filtering. Install with: pip install langchain-chroma chromadb.", metadata={"source": "rag_guide.pdf", "page": 2}),
    Document(page_content="FAISS by Meta AI is a fast in-memory vector store. It must be explicitly saved with save_local() and loaded with load_local() to persist data.", metadata={"source": "rag_guide.pdf", "page": 3}),
    Document(page_content="MMR (Maximal Marginal Relevance) retrieves diverse yet relevant documents by penalizing similarity to already-selected docs. Use lambda_mult=0.5 as default.", metadata={"source": "retrieval_guide.pdf", "page": 1}),
    Document(page_content="Tools in LangChain are functions that agents can call to interact with external systems — APIs, databases, calculators, or web search engines.", metadata={"source": "agents_guide.pdf", "page": 1}),
    Document(page_content="Conversational RAG adds chat history to plain RAG. The question is first contextualized using prior messages, then retrieval and generation proceed normally.", metadata={"source": "rag_guide.pdf", "page": 4}),
]
print(f"Knowledge base: {len(knowledge_base)} documents")

---
## 1️⃣ Simple RAG Pipeline (Manual LCEL)

In [ ]:
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# STEP 1-4: Index the knowledge base
vectorstore = Chroma.from_documents(knowledge_base, embedding=embeddings, collection_name="module07")
retriever   = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 3})

# STEP 5-6: Retrieval + Generation chain
def format_docs(docs):
    return "\n\n".join(f"[{i+1}] {d.page_content}" for i, d in enumerate(docs))

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using ONLY this context. Say 'I don't know' if unsure.\n\n{context}"),
    ("human",  "{question}")
])

simple_rag = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# Test
answer = simple_rag.invoke("What is LangGraph?")
print(f"Q: What is LangGraph?")
print(f"A: {answer}")

In [ ]:
# Test more questions
questions = [
    "How is FAISS different from Chroma?",
    "What does MMR stand for and why is it useful?",
    "What is the capital of France?",   # Out-of-scope question
]

for q in questions:
    ans = simple_rag.invoke(q)
    print(f"\n🤔 {q}")
    print(f"💡 {ans}")

---
## 2️⃣ create_retrieval_chain API

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

# Prompt requires {context} and {input}
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a LangChain expert assistant.
Answer ONLY from the provided context. If unsure, say so.

<context>
{context}
</context>"""),
    ("human", "{input}")
])

document_chain  = create_stuff_documents_chain(llm, prompt)
retrieval_chain = create_retrieval_chain(retriever, document_chain)

# Invoke — returns dict with 'input', 'context', 'answer'
result = retrieval_chain.invoke({"input": "What is LCEL?"})

print(f"Type: {type(result)}")
print(f"Keys: {list(result.keys())}")
print(f"\nAnswer: {result['answer']}")
print(f"\nContext chunks used: {len(result['context'])}")

---
## 3️⃣ Source Attribution & Streaming

In [ ]:
from pathlib import Path

def ask_with_sources(question: str):
    result = retrieval_chain.invoke({"input": question})
    
    print(f"🤔 Q: {question}")
    print(f"\n💡 A: {result['answer']}")
    
    # Source attribution
    print(f"\n📚 Sources ({len(result['context'])} chunks):")
    seen = set()
    for doc in result['context']:
        src  = doc.metadata.get('source', 'unknown')
        page = doc.metadata.get('page', '')
        key  = f"{src}:{page}"
        if key not in seen:
            seen.add(key)
            loc = f" (page {page})" if page else ""
            print(f"   • {src}{loc}")

ask_with_sources("What are the differences between FAISS and ChromaDB?")

In [ ]:
# Streaming answer token by token
print("Streaming answer:\n")
for chunk in retrieval_chain.stream({"input": "Explain the RAG pipeline steps"}):
    if "answer" in chunk:
        print(chunk["answer"], end="", flush=True)
print("\n\n[Stream complete]")

---
## 4️⃣ Conversational RAG with Manual History

In [ ]:
from langchain.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

# Prompt: contextualize question using history
contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", """Given the chat history and the latest question, 
reformulate it as a standalone question. Return as-is if already standalone."""),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)

# RAG prompt with history
conv_rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer ONLY from the context. Be concise.\n\nContext:\n{context}"),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

document_chain_conv = create_stuff_documents_chain(llm, conv_rag_prompt)
conv_rag_chain      = create_retrieval_chain(history_aware_retriever, document_chain_conv)

print("Conversational RAG chain ready!")

In [ ]:
# Multi-turn conversation
chat_history = []

def chat(question: str) -> str:
    result = conv_rag_chain.invoke({
        "input":        question,
        "chat_history": chat_history
    })
    answer = result["answer"]
    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=answer))
    return answer

# Simulate a conversation with follow-up questions
turns = [
    "What is RAG?",
    "How many steps does it have?",                       # 'it' refers to RAG
    "Which step handles the vector database?",            # refers to RAG steps
    "What tools does LangChain provide for that step?",   # 'that step' = vector db step
]

print("=" * 55)
print("Multi-turn RAG Conversation")
print("=" * 55)

for q in turns:
    print(f"\n🧑 You: {q}")
    ans = chat(q)
    print(f"🤖 Bot: {ans}")

print(f"\n📊 History: {len(chat_history)//2} turns")

---
## 5️⃣ RunnableWithMessageHistory (Session Management)

In [ ]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

# Session store
session_store: dict = {}

def get_session_history(session_id: str) -> ChatMessageHistory:
    if session_id not in session_store:
        session_store[session_id] = ChatMessageHistory()
    return session_store[session_id]

# Wrap chain with auto history management
chain_with_session = RunnableWithMessageHistory(
    conv_rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)

# Different users have independent sessions
alice = {"configurable": {"session_id": "alice"}}
bob   = {"configurable": {"session_id": "bob"}}

# Alice's conversation
print("👩 Alice:")
r1 = chain_with_session.invoke({"input": "What is LangChain?"}, config=alice)
print(f"A: {r1['answer'][:100]}...\n")

r2 = chain_with_session.invoke({"input": "What are its main components?"}, config=alice)
print(f"Q: What are its main components?")
print(f"A: {r2['answer'][:100]}...\n")

# Bob's separate conversation
print("👨 Bob:")
r3 = chain_with_session.invoke({"input": "How does FAISS work?"}, config=bob)
print(f"A: {r3['answer'][:100]}...")

# Show sessions
print(f"\n📋 Active sessions: {list(session_store.keys())}")
for sid, hist in session_store.items():
    print(f"  {sid}: {len(hist.messages)} messages")

---
## 6️⃣ Complete PDF Chatbot

In [ ]:
# Self-contained PDF Chatbot class
from langchain.chains import create_retrieval_chain, create_history_aware_retriever
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from pathlib import Path

class PDFChatbot:
    """RAG Chatbot with conversational memory."""

    def __init__(self, docs, collection_name="chatbot"):
        """Initialize with a list of Documents."""
        self.chat_history = []
        self.llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

        # Build index
        splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
        chunks   = splitter.split_documents(docs)
        store    = Chroma.from_documents(chunks, embeddings, collection_name=collection_name)
        retriever = store.as_retriever(search_type="mmr", search_kwargs={"k": 3, "fetch_k": 10})

        # Build chain
        ctx_prompt = ChatPromptTemplate.from_messages([
            ("system", "Reformulate as standalone question if needed, else return as-is."),
            MessagesPlaceholder("chat_history"),
            ("human", "{input}"),
        ])
        rag_prompt = ChatPromptTemplate.from_messages([
            ("system", "Answer ONLY from context. Say 'I don't know' if unsure.\n\nContext:\n{context}"),
            MessagesPlaceholder("chat_history"),
            ("human", "{input}"),
        ])

        hist_retriever = create_history_aware_retriever(self.llm, retriever, ctx_prompt)
        doc_chain      = create_stuff_documents_chain(self.llm, rag_prompt)
        self.chain     = create_retrieval_chain(hist_retriever, doc_chain)
        print(f"✅ Chatbot ready ({len(chunks)} chunks indexed)")

    def chat(self, question: str) -> str:
        result = self.chain.invoke({"input": question, "chat_history": self.chat_history})
        answer = result["answer"]
        sources = list({d.metadata.get("source", "?") for d in result["context"]})
        self.chat_history.extend([HumanMessage(content=question), AIMessage(content=answer)])
        return answer, sources

    def reset(self):
        self.chat_history = []

# Initialize with our shared knowledge base
bot = PDFChatbot(knowledge_base, collection_name="module07-chatbot")

In [ ]:
# Full conversation demo
conversation = [
    "What is LangGraph?",
    "How does it differ from LangChain?",
    "Can you give me an example use case for it?",
    "What about LangSmith, is it related?",
]

print("🤖 PDF Chatbot Demo")
print("=" * 50)

for q in conversation:
    answer, sources = bot.chat(q)
    print(f"\n🧑 {q}")
    print(f"🤖 {answer}")
    print(f"   📎 {', '.join(sources)}")

---
## ✅ Summary — RAG Patterns

| Pattern | When to Use | Key Components |
|---|---|---|
| Simple RAG | One-shot Q&A | `retriever \| format_docs \| prompt \| llm` |
| `create_retrieval_chain` | Production Q&A | `create_stuff_documents_chain` + `create_retrieval_chain` |
| Conversational RAG | Multi-turn chatbot | `create_history_aware_retriever` + `MessagesPlaceholder` |
| Session RAG | Multi-user app | `RunnableWithMessageHistory` + session store |
| Streaming RAG | Token-by-token display | `.stream()` + extract `chunk["answer"]` |

**Next**: [08 — Tools & Tool Calling](../08_tools_and_tool_calling/theory.md)